In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
PRIORITAS = ['Riau', 'Jambi', 'Sumatera Selatan', 'Kalimantan Barat', 'Kalimantan Tengah', 'Kalimantan Selatan']

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'],
                    usecols=['tanggal', 'GID_2', 'NAME_1', 'tahun', 'bulan', 'titik_panas', 'frp_total'])
z = np.load(HASIL / 'prob_final_nasional.npz')
assert len(z['PC']) == len(tabel), 'urutan baris tidak sama dengan tabel'
tabel['p_K1'] = z['PC'][:, 1]      # model C (ERA5 00 UTC, sebelum jendela hujan): dipakai untuk skor
tabel['p_K1_D'] = z['PD'][:, 1]    # model D (awan serentak): pembanding saja, tidak masuk skor

tabel = tabel.sort_values(['GID_2', 'tanggal']).reset_index(drop=True)
tabel['api_7h'] = (tabel.groupby('GID_2')['titik_panas']
                   .transform(lambda s: s.shift(1).rolling(7, min_periods=4).sum()))

p = tabel.copy()
p['r_awan'] = p.groupby('tanggal')['p_K1'].rank(pct=True)
p['r_api'] = p.groupby('tanggal')['api_7h'].rank(pct=True)
p['skor'] = p['r_awan'] * p['r_api']
p['enam_prov'] = p['NAME_1'].isin(PRIORITAS)
p.to_csv(HASIL / 'prioritas_harian.csv', index=False)

hari = p.loc[p['api_7h'].notna(), 'tanggal'].max()
h = p[p['tanggal'] == hari].sort_values('skor', ascending=False)
cols = ['GID_2', 'NAME_1', 'p_K1', 'api_7h', 'skor']
print('tanggal:', hari.date(), '\nNASIONAL')
print(h[cols].head(10).round(3).to_string(index=False))
print('\nENAM PROVINSI')
print(h[h['enam_prov']][cols].head(10).round(3).to_string(index=False))

# tes lompatan FIRMS SP -> NRT (1 Jul 2026) untuk kab/kota teratas
top = h['GID_2'].head(10).tolist()
x = tabel[tabel['GID_2'].isin(top)].copy()
x['bln'] = x['tanggal'].dt.to_period('M')
rata = x.pivot_table(index='GID_2', columns='bln', values='titik_panas', aggfunc='mean')
kol = [pd.Period(f'2025-{m:02d}') for m in (5, 6, 7, 8)] + [pd.Period(f'2026-{m:02d}') for m in (5, 6, 7, 8, 9)]
print('\nRATA-RATA TITIK PANAS HARIAN PER BULAN')
print(rata.reindex(columns=kol).round(1).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tanggal: 2026-10-14 
NASIONAL
      GID_2       NAME_1  p_K1  api_7h  skor
IDN.22.10_1  Papua Barat 0.359   483.0 0.866
 IDN.18.5_1 Maluku Utara 0.367   195.0 0.850
 IDN.22.1_1  Papua Barat 0.321  1023.0 0.832
 IDN.18.7_1 Maluku Utara 0.397    75.0 0.818
 IDN.22.7_1  Papua Barat 0.336   268.0 0.814
 IDN.18.3_1 Maluku Utara 0.364   116.0 0.811
 IDN.18.4_1 Maluku Utara 0.315   620.0 0.803
 IDN.23.8_1        Papua 0.413    55.0 0.789
IDN.23.28_1        Papua 0.447    45.0 0.774
IDN.19.10_1       Maluku 0.310   235.0 0.768

ENAM PROVINSI
      GID_2            NAME_1  p_K1  api_7h  skor
IDN.12.11_1  Kalimantan Barat 0.383    23.0 0.684
 IDN.14.6_1 Kalimantan Tengah 0.220  2506.0 0.656
 IDN.8.10_1             Jambi 0.233   515.0 0.653
  IDN.8.6_1             Jambi 0.225   343.0 0.633
 IDN.14.8_1 Kalimantan Tengah 0.159  1779.0 0.608
IDN.12.14_1  Kalimantan Barat 0